In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from opacus import PrivacyEngine
import gc
import numpy as np
import torch.nn.functional as F
from DP_SGD_Materials.compute_privacy_sgm import compute_dp_sgd_privacy



transform_svhn = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Grayscale(),
])

svhn_dataset  = datasets.SVHN(root='./data', split="train", transform=transform_svhn, download=True)
svhn_loader  = DataLoader(dataset=svhn_dataset, batch_size=64, shuffle=False)

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), 
    transforms.ToTensor(),
])

cifar_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)

private_subset = cifar_dataset

np.random.seed(42)
svhn_indices = np.random.choice(len(svhn_dataset), size=2000, replace=False)
public_subset = Subset(svhn_dataset, svhn_indices)

cifar_test_data = datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)
test_loader = DataLoader(dataset=cifar_test_data, batch_size=64, shuffle=False)
train_loader_private = DataLoader(dataset=private_subset, batch_size=64, shuffle=True)
train_loader_public = DataLoader(dataset=public_subset, batch_size=64, shuffle=True)
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv1 = nn.Conv2d(1,6, kernel_size=5, stride=1, padding=2)
        self.tanh1 = nn.Tanh()
        self.pool1 = nn.AvgPool2d(kernel_size=2,  stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1)
        self.tanh2 = nn.Tanh()
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1)
        self.tanh3 = nn.Tanh()
        self.fc1 = nn.Linear(480, 84)
        self.tanh4 = nn.Tanh()
        self.fc2 = nn.Linear(84, 10)
    def forward(self, x):
        x = self.pool1(self.tanh1(self.conv1(x)))
        x = self.pool2(self.tanh2(self.conv2(x)))
        x = self.tanh3(self.conv3(x))
        x = x.view(-1, 480)
        x = self.tanh4(self.fc1(x))
        x = self.fc2(x)
        return x
    
model = Model()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

privacy_engine = PrivacyEngine()

gc.collect()
epochs_1 = 5
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []
for epoch in range(epochs_1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader_public:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs_1}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)
    gc.collect()

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    gc.collect()
# from opacus.noise_generator import GaussianNoiseGenerator
from opacus.validators import ModuleValidator

epochs_2 = 10
train_losses_2 = []
train_accuracies_2 = []
test_losses_2 = []
test_accuracies_2 = []
model.train()
target_delta = 1e-5
# noise_generator = GaussianNoiseGenerator(
#     noise_multiplier=sigma,
#     sample_rate=sample_rate,  # Required
#     device=device,            # "cuda" or "cpu"
# )
model, optimizer, train_loader_private = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader_private,
    epochs=10,
    target_delta=target_delta,
    max_grad_norm=1,
    noise_multiplier=0.0
)
for epoch in range(epochs_2):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader_private:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs_2}]')
    print("Training loss", (running_loss / total))
    train_losses_2.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies_2.append(correct * 100 / total)
    gc.collect()

    # epsilon = privacy_engine.get_epsilon(delta=target_delta)
    # print(f"Privacy Budget (epsilon, delta): ({epsilon:.2f}, {target_delta})")

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss / total))
    test_losses_2.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies_2.append(correct * 100 / total)

    gc.collect()
    model.eval()
    correct = 0
    total = 0
    eval_loss = 0.0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            eval_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

print(f"Test Loss: {eval_loss / total:.4f}, Accuracy: {100 * correct / total:.2f}%")



/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


Epoch [1/5]
Training loss 2.2673594245910644
Training accuracy 15.9 %
Testing loss 0.4534718849182129
Testing accuracy 10.0 %
Epoch [2/5]
Training loss 2.222499040603638
Training accuracy 19.35 %
Testing loss 0.44449980812072754
Testing accuracy 10.0 %
Epoch [3/5]
Training loss 2.2220358238220217
Training accuracy 19.35 %
Testing loss 0.4444071647644043
Testing accuracy 10.0 %
Epoch [4/5]
Training loss 2.2205020198822023
Training accuracy 19.35 %
Testing loss 0.4441004039764404
Testing accuracy 10.0 %
Epoch [5/5]
Training loss 2.221244905471802
Training accuracy 19.35 %
Testing loss 0.44424898109436034
Testing accuracy 10.0 %
Epoch [1/10]
Training loss 2.216405460861525
Training accuracy 18.741145517101803 %
Testing loss 10.951259382116795
Testing accuracy 24.93 %
Epoch [2/10]
Training loss 2.1296261102596556
Training accuracy 25.537050446536327 %
Testing loss 10.587649169766904
Testing accuracy 26.31 %
Epoch [3/10]
Training loss 2.0839636501058103
Training accuracy 27.344760994644528 